# Walkthrough: how `generate()` and `evaluate()` work

An interactive teardown of Section 6 of the fine-tuning notebooks. We build intuition for:
1. **Chat templating** — turning a query into the model's expected prompt format.
2. **Tokenization** — text → token IDs.
3. **Batching + left padding + attention masks** — making sequences the same length.
4. **One forward pass** — logits → next-token probabilities → the greedy pick.
5. **The autoregressive loop** — appending generated tokens one at a time.
6. **`parse_json` + `evaluate`** — scoring the output.

**No GPU needed** — Qwen2.5-0.5B is tiny and we only generate a few tokens. Runs on CPU in seconds.

## Setup

In [ ]:
!pip install -q -U transformers torch pandas

In [ ]:
import json
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # base model is fine; we're studying mechanics, not quality
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Use float32 everywhere for a clean, deterministic walkthrough.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
model.eval()

SYSTEM_PROMPT = (
    "You extract structured information from airline-travel queries. "
    "Respond with ONLY a JSON object with exactly these keys: "
    'intent, from_city, to_city, depart_date, depart_time, airline, round_trip, class_type. '
    "'intent' is a list whose values are a subset of "
    '["flight", "airfare", "ground_service", "airline"]. '
    "Every other field is a string copied from the query, or null if not mentioned. "
    "Output only the JSON, no extra text."
)

def to_prompt(query):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}]

print("device:", device)
print("pad_token:", tokenizer.pad_token, "| id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token, "| id:", tokenizer.eos_token_id)

## 1. Chat templating

A chat model doesn't see raw text — it expects a specific format with role markers. Qwen uses
**ChatML**: `<|im_start|>role ... <|im_end|>`. `apply_chat_template` builds that string from our
list of messages. `add_generation_prompt=True` appends the opening `<|im_start|>assistant` so the
model knows it's its turn to speak.

In [ ]:
query = "flights from boston to denver on monday morning"
prompt_str = tokenizer.apply_chat_template(to_prompt(query), tokenize=False,
                                           add_generation_prompt=True)
print(prompt_str)

Notice the structure: a `system` block (our instructions), a `user` block (the query), and a
dangling `<|im_start|>assistant` at the end. The model will generate what comes after that.

## 2. Tokenization

The model operates on integer **token IDs**, not characters. Tokenizing the prompt gives
`input_ids` (the IDs) and an `attention_mask` (all 1s for a single, unpadded sequence).

In [ ]:
enc1 = tokenizer(prompt_str, return_tensors="pt")
print("input_ids shape:", enc1["input_ids"].shape)
print("first 20 ids:", enc1["input_ids"][0, :20].tolist())

# Map a few IDs back to their token strings to see the pieces.
ids = enc1["input_ids"][0]
toks = tokenizer.convert_ids_to_tokens(ids)
pd.DataFrame({"position": range(len(ids)), "id": ids.tolist(), "token": toks}).head(25)

The special tokens (`<|im_start|>`, `<|im_end|>`) are single IDs. Note `Ġ` (or a leading space
marker) on many tokens — that's how byte-level BPE encodes word boundaries.

## 3. Batching, left padding, and the attention mask

To process several queries at once, every sequence in the batch must be the **same length**.
Shorter ones get filled with the `pad` token. Two key ideas:
- The **attention mask** marks real tokens (`1`) vs padding (`0`) so the model ignores padding.
- For **generation** we pad on the **left**, so every prompt *ends* at the same right edge. New
  tokens are appended on the right, right after each prompt's real last token. (Right-padding
  would wedge pad tokens between the prompt and the generated text — wrong for decoder generation.)

In [ ]:
queries = [
    "cheapest airfare from tacoma to orlando",            # shorter
    "flights from boston to denver on monday morning",    # longer
]
prompts = [tokenizer.apply_chat_template(to_prompt(q), tokenize=False,
                                         add_generation_prompt=True) for q in queries]

tokenizer.padding_side = "left"
batch = tokenizer(prompts, return_tensors="pt", padding=True)
print("batched input_ids shape:", batch["input_ids"].shape, "(rows=queries, cols=padded length)")

# Look at the LAST 12 columns of each row: see padding land on the LEFT and the real tokens align right.
for r in range(2):
    print(f"\nrow {r} last 12 ids :", batch['input_ids'][r, -12:].tolist())
    print(f"row {r} last 12 mask:", batch['attention_mask'][r, -12:].tolist())
# And the FIRST 12 columns of the shorter row should be pad tokens (mask 0).
print("\nshorter row (row 0) FIRST 12 ids :", batch['input_ids'][0, :12].tolist())
print("shorter row (row 0) FIRST 12 mask:", batch['attention_mask'][0, :12].tolist())
print("pad_token_id =", tokenizer.pad_token_id)

In [ ]:
# Decode the shorter row to literally see the <pad> tokens prepended on the left.
print(repr(tokenizer.decode(batch["input_ids"][0])[:120]), "...")

### Two different masks — don't confuse them
- **Padding mask** (the `attention_mask` we pass): says which positions are real vs padding.
- **Causal mask** (applied *inside* the model, we don't pass it): a lower-triangular mask that
  stops each position from attending to *future* positions — that's what makes it a left-to-right
  language model. Illustration below (1 = can attend, 0 = blocked).

In [ ]:
n = 6
causal = torch.tril(torch.ones(n, n, dtype=int))
print("causal mask (row = query position, col = key position):")
print(pd.DataFrame(causal.numpy(), index=[f"pos{i}" for i in range(n)],
                   columns=[f"pos{i}" for i in range(n)]))

## 4. One forward pass: logits → next token

A single forward pass returns **logits** of shape `(batch, seq_len, vocab_size)` — a score for
every vocabulary token at every position. For generation we only care about the **last position**
(`logits[:, -1, :]`): that's the model's prediction for the *next* token. Greedy decoding just
takes the `argmax`. Let's also softmax to see the top candidates as probabilities.

In [ ]:
single = tokenizer(prompts[1], return_tensors="pt").to(device)  # the longer query
with torch.no_grad():
    out = model(**single)
print("logits shape:", tuple(out.logits.shape), "= (batch, seq_len, vocab_size)")

last_logits = out.logits[0, -1, :]            # prediction for the next token
probs = torch.softmax(last_logits, dim=-1)
top = torch.topk(probs, 5)
pd.DataFrame({
    "token_id": top.indices.tolist(),
    "token": tokenizer.convert_ids_to_tokens(top.indices),
    "probability": [f"{p:.3f}" for p in top.values.tolist()],
})

The top row (highest probability) is what greedy decoding picks as the first generated token —
almost certainly `{`, the start of the JSON object.

## 5. The autoregressive loop (by hand)

Generation = repeat the forward pass, each time **appending** the chosen token and extending the
attention mask by one, until we hit the end-of-sequence token (or a max). Watch the text grow.

In [ ]:
ids = single["input_ids"].clone()
mask = single["attention_mask"].clone()
start_len = ids.shape[1]

for step in range(40):
    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=mask).logits
    next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # greedy pick, shape (1, 1)
    ids = torch.cat([ids, next_id], dim=1)                    # APPEND the new token
    mask = torch.cat([mask, torch.ones_like(next_id)], dim=1) # extend mask by one real token
    tok = tokenizer.convert_ids_to_tokens(next_id[0])[0]
    if step < 12:
        sofar = tokenizer.decode(ids[0, start_len:], skip_special_tokens=True)
        print(f"step {step:2d} | +token {next_id.item():>6} {tok!r:12} | so far: {sofar!r}")
    if next_id.item() == tokenizer.eos_token_id:
        print(f"step {step:2d} | hit EOS, stop")
        break

print("\nfull generation:", tokenizer.decode(ids[0, start_len:], skip_special_tokens=True))

Each step does a full forward pass over the whole sequence and reads off the last position. (Real
implementations cache past keys/values so they don't recompute everything each step, but the logic
is exactly this.) `model.generate(...)` wraps this loop for us — let's confirm it matches.

In [ ]:
with torch.no_grad():
    gen = model.generate(**single, max_new_tokens=40, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
# Slice off the prompt — keep only the NEW tokens (this is the gen[j][input_len:] step).
new_tokens = gen[0, start_len:]
print("generate() output:", tokenizer.decode(new_tokens, skip_special_tokens=True))
print("\nmatches our hand loop:",
      tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
      == tokenizer.decode(ids[0, start_len:], skip_special_tokens=True).strip())

## 6. The real `generate()` — same thing, batched

Here is the exact helper from the fine-tuning notebooks. Every line should now look familiar:
set left padding → build prompts → tokenize the batch → `model.generate` → slice off the prompt
(`gen[j][input_len:]`) → decode.

In [ ]:
@torch.no_grad()
def generate(queries, max_new_tokens=128, batch_size=16):
    model.eval()
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    out = []
    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        prompts = [tokenizer.apply_chat_template(to_prompt(q), tokenize=False,
                                                 add_generation_prompt=True) for q in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=384).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]   # drop the prompt tokens
            out.append(tokenizer.decode(new, skip_special_tokens=True))
    tokenizer.padding_side = prev_side
    return out

for q, txt in zip(queries, generate(queries, max_new_tokens=64)):
    print("Q   :", q)
    print("pred:", txt.strip())
    print("-" * 70)

> Because this is the **base** (not fine-tuned) model, the JSON may be imperfect — that's fine,
> we're studying the mechanics. After fine-tuning, these become clean structured objects.

## 7. `parse_json` — pulling JSON out of the text

The model sometimes adds stray text around the JSON. `parse_json` grabs the substring from the
first `{` to the last `}` and tries to parse it; returns `None` if that fails (invalid JSON).

In [ ]:
def parse_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e == -1 or e < s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None

print(parse_json('Sure! {"intent": ["flight"], "from_city": "boston"} hope that helps'))
print(parse_json('no json here'))           # -> None
print(parse_json('{"broken": '))            # -> None (invalid)

## 8. `evaluate` — scoring predictions vs gold

For each example: parse the prediction; if valid, compare to the gold record.
- **intent**: compared as a **set** (order-independent).
- **each field**: case-insensitive, whitespace-trimmed exact-match (`_norm`).
- **exact match**: every field *and* intent correct.
Counts are accumulated and turned into percentages. Let's run it on tiny hand-made data.

In [ ]:
SCHEMA_FIELDS = ["from_city", "to_city", "depart_date", "depart_time",
                 "airline", "round_trip", "class_type"]

def _norm(v):
    return v.strip().lower() if isinstance(v, str) else v

# Two fake examples: (gold, model_text)
examples = [
    ({"intent": ["flight"], "from_city": "boston", "to_city": "denver",
      "depart_date": "monday", "depart_time": "morning", "airline": None,
      "round_trip": None, "class_type": None},
     '{"intent": ["flight"], "from_city": "Boston", "to_city": "denver", "depart_date": "monday", "depart_time": "morning", "airline": null, "round_trip": null, "class_type": null}'),
    ({"intent": ["airfare"], "from_city": "tacoma", "to_city": "orlando",
      "depart_date": None, "depart_time": None, "airline": None,
      "round_trip": None, "class_type": None},
     'oops not json'),
]

n = len(examples)
valid = exact = intent_ok = 0
field_ok = {f: 0 for f in SCHEMA_FIELDS}
for gold, text in examples:
    pred = parse_json(text)
    print("gold:", gold)
    print("pred:", pred)
    if pred is None:
        print("  -> invalid JSON, counts as a miss on everything\n"); continue
    valid += 1
    gi, pi = set(gold["intent"]), set(pred.get("intent", []))
    all_ok = gi == pi
    intent_ok += all_ok
    print(f"  intent {gi} vs {pi} -> {'OK' if gi==pi else 'MISS'}")
    for f in SCHEMA_FIELDS:
        ok = _norm(pred.get(f)) == _norm(gold[f])
        field_ok[f] += ok
        if not ok:
            all_ok = False
            print(f"  field {f}: {pred.get(f)!r} vs {gold[f]!r} -> MISS")
    exact += all_ok
    print(f"  exact match for this example: {all_ok}\n")

print(f"valid JSON : {valid/n:.0%}")
print(f"exact match: {exact/n:.0%}")
print(f"intent acc : {intent_ok/n:.0%}")
for f in SCHEMA_FIELDS:
    print(f"  {f:<12}: {field_ok[f]/n:.0%}")

Note in the first example `"Boston"` (gold `"boston"`) still counts as correct thanks to `_norm`
lowercasing — but it is otherwise **strict exact-match**, so a partial date like `"june"` vs
`"june 12"` would be a miss. The second example is invalid JSON, so it misses everything and drags
down every metric — which is why `valid JSON %` is the first thing to look at.

## Recap
- **Prompt**: messages → ChatML string (`apply_chat_template`, `add_generation_prompt`).
- **Tokenize**: string → `input_ids` + `attention_mask`.
- **Batch**: **left**-pad so prompts end at the right edge; mask marks real vs pad tokens.
- **Generate**: forward pass → last-position logits → greedy `argmax` → append → repeat until EOS.
- **Slice**: keep only tokens after the prompt; decode to text.
- **Score**: `parse_json` → set-compare intent + `_norm` exact-match per field → percentages.